## Radial roughness elements positioning

Positions roughness fins radially above a set of STL surfaces (terrain + loft).
Each fin is oriented so its face normal points radially outward from the center.

In [1]:
import numpy as np
import pathlib

element_height = 2
element_width  = 6

r_start              = 300.0
r_end                = 3000.0
radial_spacing       = 16.0
arc_spacing          = 40.0
ring_offset_distance = element_width / 2

center = np.array([0.0, 0.0])

surface_paths = [
    pathlib.Path("fixtures/tests/loft/complex_terrain.stl"),
    pathlib.Path("output/complex_loft/longer_loft/loft.stl"),
]

output_path = pathlib.Path("output/roughness_radial.stl")

Load surfaces and build Z interpolator from all surface vertices

In [2]:
from lnas import LnasFormat
from scipy.interpolate import LinearNDInterpolator

all_verts_list = []
for path in surface_paths:
    geom = LnasFormat.from_file(path).geometry
    all_verts_list.append(geom.vertices.astype(np.float64))

all_verts = np.unique(np.concatenate(all_verts_list), axis=0)

z_interp = LinearNDInterpolator(all_verts[:, :2], all_verts[:, 2])

print(f"Surface vertices (deduplicated): {len(all_verts)}")
print(f"Z range: {all_verts[:, 2].min():.2f} to {all_verts[:, 2].max():.2f}")

Surface vertices (deduplicated): 34298
Z range: -1.17 to 206.20


Generate radial grid positions

Fins per ring = floor(2*pi*r / arc_spacing), keeping arc-length spacing
approximately constant across rings. Alternating rings are offset by
ring_offset_distance (meters) -> angle = offset_distance / r, so the
physical stagger is constant across rings.

In [3]:
positions = []

rings = np.arange(r_start, r_end + radial_spacing * 0.5, radial_spacing)
for ring_idx, r in enumerate(rings):
    n_fins = max(1, int(2.0 * np.pi * r / arc_spacing))
    base_angle = (ring_offset_distance / r) * (ring_idx % 2)
    angles = np.linspace(0.0, 2.0 * np.pi, n_fins, endpoint=False) + base_angle
    for theta in angles:
        x = center[0] + r * np.cos(theta)
        y = center[1] + r * np.sin(theta)
        positions.append((x, y, theta))

positions = np.array(positions)
print(f"Rings: {len(rings)}   Total fin positions: {len(positions)}")

Rings: 170   Total fin positions: 44029


Sample Z height at each fin position via interpolation

In [4]:
z_heights = z_interp(positions[:, 0], positions[:, 1])

valid_mask = ~np.isnan(z_heights)
print(f"Positions with valid Z: {valid_mask.sum()} / {len(positions)}")

Positions with valid Z: 43558 / 44029


Build fins

Base fin (before rotation):
  V0=[0,0,0]  V1=[0,w,0]  V2=[0,w,h]  V3=[0,0,h]  face normal [-1,0,0]

Rotating by theta around Z gives face normal [cos(theta), sin(theta), 0]
(outward radial). The fin is centered tangentially at (x_pos, y_pos, z_surface).

In [5]:
base_verts = np.array([
    [0.0, 0.0,           0.0],
    [0.0, element_width, 0.0],
    [0.0, element_width, element_height],
    [0.0, 0.0,           element_height],
], dtype=np.float64)

all_triangles = []
all_normals   = []

for i, (x_pos, y_pos, theta) in enumerate(positions):
    if not valid_mask[i]:
        continue

    cos_t = np.cos(theta)
    sin_t = np.sin(theta)

    R = np.array([
        [cos_t, -sin_t, 0.0],
        [sin_t,  cos_t, 0.0],
        [0.0,    0.0,   1.0],
    ])
    rotated = (R @ base_verts.T).T

    translation = np.array([
        x_pos + (element_width / 2.0) * sin_t,
        y_pos - (element_width / 2.0) * cos_t,
        z_heights[i],
    ])
    verts = (rotated + translation).astype(np.float32)

    n = np.array([cos_t, sin_t, 0.0], dtype=np.float32)

    all_triangles.append(verts[[0, 1, 2]])
    all_triangles.append(verts[[0, 2, 3]])
    all_normals.append(n)
    all_normals.append(n)

all_triangles = np.array(all_triangles, dtype=np.float32)
all_normals   = np.array(all_normals,   dtype=np.float32)

print(f"Generated {len(all_triangles)} triangles ({len(all_triangles) // 2} fins)")

Generated 87116 triangles (43558 fins)


Export

In [6]:
from lnas import LnasFormat

geom = LnasFormat.from_triangles(triangles=all_triangles, normals=all_normals).geometry
output_path.parent.mkdir(parents=True, exist_ok=True)
geom.export_stl(output_path)
print(f"Exported to {output_path}")

Exported to output/roughness_radial.stl
